In [ ]:
# Cell 1: 导入依赖 & 超参数设置

import os, math, random, copy
import re
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import librosa
import jieba
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# 路径：改成本机实际路径
DATA_ROOT    = r"E:\EATD-Corpus"
LEXICON_ROOT = r"E:\lexicons"

# 随机划分参数
SPLIT_RS     = 7
MODEL_SEED   = 42
CLASS_WEIGHT = [1.0, 4.0]
FIXED_THR    = 0.45
THRESHOLDS   = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

# 模型 & 训练超参
L = 128; D = 128; SDS_THR = 53
NUM_EPOCHS = 150; BATCH_SIZE = 8
LR = 5e-4; WEIGHT_DECAY = 5e-4
DROPOUT_P = 0.1; PATIENCE = 25

CHECKPOINT_PATH = "sag_fusion_best.pt"

device = (torch.device("cuda") if torch.cuda.is_available() else
          torch.device("mps") if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else
          torch.device("cpu"))

print(f"device      : {device}")
print(f"DATA_ROOT   : {DATA_ROOT}")


In [ ]:
# Cell 2: 数据集遍历 & 划分

def read_num(path):
    if not os.path.exists(path): return np.nan
    for enc in ["utf-8", "gbk"]:
        try:
            with open(path, "r", encoding=enc) as f: return float(f.read().strip())
        except: continue
    return np.nan

def build_manifest(data_root, split_rs=42):
    rows = []
    for name in sorted(os.listdir(data_root)):
        subdir = os.path.join(data_root, name)
        if not os.path.isdir(subdir) or not (name.startswith("t_") or name.startswith("v_")): continue
        row = {"subject_id": name}
        for seg, short in [("negative","neg"),("neutral","neu"),("positive","pos")]:
            row[f"{short}_wav"] = os.path.join(subdir, f"{seg}_out.wav")
            row[f"{short}_txt"] = os.path.join(subdir, f"{seg}.txt")
        lnew = read_num(os.path.join(subdir, "new_label.txt"))
        row["label_new"] = lnew
        row["y_bin"] = int(lnew >= SDS_THR) if not np.isnan(lnew) else np.nan
        rows.append(row)
    mf = pd.DataFrame(rows).dropna(subset=["y_bin"]).reset_index(drop=True)
    mf["y_bin"] = mf["y_bin"].astype(int)
    ids = mf["subject_id"].values; y = mf["y_bin"].values
    tr_ids, tmp, _, ytmp = train_test_split(ids, y, test_size=0.40, random_state=split_rs, stratify=y)
    va_ids, te_ids = train_test_split(tmp, test_size=0.50, random_state=split_rs, stratify=ytmp)
    mf["split"] = "ignore"
    mf.loc[mf["subject_id"].isin(tr_ids), "split"] = "train"
    mf.loc[mf["subject_id"].isin(va_ids), "split"] = "val"
    mf.loc[mf["subject_id"].isin(te_ids), "split"] = "test"
    mf = mf[mf["split"] != "ignore"].reset_index(drop=True)
    return mf

manifest = build_manifest(DATA_ROOT, split_rs=SPLIT_RS)

print("manifest basic info")
print(f"  shape: {manifest.shape}")
display(manifest.head())
print("cell 2 over")


In [ ]:

# Cell 3: 情感词典加载 & 文本打分函数

def _rl(path):
    if not os.path.exists(path): return []
    for enc in ["utf-8", "gbk"]:
        try:
            with open(path, "r", encoding=enc) as f: return f.readlines()
        except: continue
    return []

def lwl(path):
    return {l.strip().split()[0] for l in _rl(path)
            if l.strip() and not l.startswith("#") and l.strip().split()}

def lwsd(path, ds=1.0):
    d = {}
    for l in _rl(path):
        p = l.strip().split()
        if not p or p[0].startswith("#"): continue
        try: d[p[0]] = float(p[1]) if len(p) >= 2 else ds
        except: d[p[0]] = ds
    return d

ntusd_pos = lwl(os.path.join(LEXICON_ROOT, "ntusd-positive.txt"))
ntusd_neg = lwl(os.path.join(LEXICON_ROOT, "ntusd-negative.txt"))
neg_words = lwl(os.path.join(LEXICON_ROOT, "negation.txt"))
stops     = lwl(os.path.join(LEXICON_ROOT, "stopwords_hit.txt"))
intensity = lwsd(os.path.join(LEXICON_ROOT, "intensity.txt"), 1.5)
boson     = lwsd(os.path.join(LEXICON_ROOT, "boson.txt"), 0.0)

TURN = {"但是", "但", "然而", "不过", "可是", "却"}
_SP  = re.compile(r"[。！？!?…\n]+")
TAU  = 4.0

def split_sents(t):
    return [p.strip() for p in _SP.split(t.strip()) if p.strip()] if t else []

def score_clause(toks):
    s = 0.0
    for i, w in enumerate(toks):
        if w in stops: continue
        b = boson.get(w, 0.0) + (1.0 if w in ntusd_pos else 0.0) - (1.0 if w in ntusd_neg else 0.0)
        if b == 0: continue
        win = toks[max(0, i-3):i]; d = 1.0; nf = 1.0
        for u in win:
            if u in intensity: d *= intensity[u]
            if u in neg_words: nf *= -1
        s += b * d * nf
    return s

def score_sent(t):
    if not t or not t.strip(): return 0.0
    toks = jieba.lcut(t); cls = []; cur = []
    for w in toks:
        if w in TURN:
            if cur: cls.append(cur); cur = []
        else: cur.append(w)
    if cur: cls.append(cur)
    if not cls: return 0.0
    ws = [1.0] * len(cls); ws[-1] = 1.5
    return sum(w * score_clause(c) for w, c in zip(ws, cls)) / (sum(ws) + 1e-6)

def bucket_pool(S, L_):
    C, M = S.shape; P = torch.zeros(C, L_); cnt = torch.zeros(L_, dtype=torch.long)
    if M > 0:
        for i in range(M):
            b = min(int(i * L_ / M), L_ - 1); P[:, b] += S[:, i]; cnt[b] += 1
        for b in range(L_):
            if cnt[b] > 0: P[:, b] /= cnt[b].float()
    return P, cnt > 0

def enc_txt_raw(path):
    raw = ""
    if os.path.exists(path):
        for enc in ["utf-8", "gbk"]:
            try:
                with open(path, "r", encoding=enc) as f: raw = f.read(); break
            except: continue
    sents = split_sents(raw) or [""]
    S = torch.zeros(4, len(sents))
    for j, s in enumerate(sents):
        x = math.tanh(score_sent(s) / TAU)
        S[0,j] = max(-x, 0); S[1,j] = 1 - min(1, abs(x)); S[2,j] = max(x, 0); S[3,j] = x
    return bucket_pool(S, L)

print(f"pos={len(ntusd_pos)}, neg={len(ntusd_neg)}, neg_words={len(neg_words)}, "
      f"intensity={len(intensity)}, boson={len(boson)}")


In [ ]:
# Cell 4: TextEncoder

class TextEncoder:
    def __init__(self):
        self.mean = torch.zeros(4); self.std = torch.ones(4); self._fit = False
        self.proj = nn.Linear(4, D, bias=True)   # 不加入 optimizer

    def fit(self, mf_tr):
        sc = torch.zeros(4, dtype=torch.float64); ssq = sc.clone(); cnt = sc.clone()
        for _, row in mf_tr.iterrows():
            for col in ["neg_txt", "neu_txt", "pos_txt"]:
                P, mask = enc_txt_raw(row[col])
                if mask.any():
                    for c in range(4):
                        v = P[c, mask]; sc[c] += v.sum(); ssq[c] += (v**2).sum(); cnt[c] += v.numel()
        cnt = cnt.clamp(1); m = sc / cnt; v = (ssq / cnt - m**2).clamp(1e-6)
        self.mean = m.float(); self.std = v.sqrt().float(); self._fit = True
        for c, name in enumerate(["neg","neu","pos","con"]):
            print(f"  ch{c}({name}): mean={self.mean[c]:.4f}, std={self.std[c]:.4f}")

    def encode(self, row):
        TL, ML = [], []
        for col in ["neg_txt", "neu_txt", "pos_txt"]:
            P, mask = enc_txt_raw(row[col])
            if self._fit: P = (P - self.mean.view(-1,1)) / (self.std.view(-1,1) + 1e-6)
            t = self.proj(P.T).T; TL.append(t); ML.append(mask)
        return torch.stack(TL), torch.stack(ML).bool()

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(MODEL_SEED)
text_encoder = TextEncoder()
mf_tr = manifest[manifest["split"] == "train"].reset_index(drop=True)
print("Fitting TextEncoder ...")
text_encoder.fit(mf_tr)
print("cell 4 over")


In [ ]:
# Cell 5: AudioEncoder

def load_wav(path, sr=22050):
    if not os.path.exists(path): return np.zeros(sr, dtype=np.float32), sr
    try:
        w, _ = librosa.load(path, sr=sr)
        return np.asarray(w, dtype=np.float32).flatten() or np.zeros(sr, dtype=np.float32), sr
    except: return np.zeros(sr, dtype=np.float32), sr

def ext_feat(wav, sr=22050):
    if wav is None or wav.size < 10: return np.zeros((8, 1), dtype=np.float32)
    mf = librosa.feature.mfcc(y=wav, sr=sr, n_mfcc=20)
    ch = librosa.feature.chroma_stft(y=wav, sr=sr, n_chroma=12)
    ml = librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=40)
    co = librosa.feature.spectral_contrast(y=wav, sr=sr)
    T  = min(mf.shape[1], ch.shape[1], ml.shape[1], co.shape[1])
    if T <= 0: return np.zeros((8, 1), dtype=np.float32)
    def ms(f): return f[:,:T].mean(0), f[:,:T].std(0)
    a, b = ms(mf); c, d_ = ms(ch); e, f_ = ms(ml); g, h = ms(co)
    return np.stack([a, b, c, d_, e, f_, g, h]).astype(np.float32)

class AudioEncoder:
    def __init__(self):
        self.mean = torch.zeros(8); self.std = torch.ones(8); self._fit = False
        self.conv = nn.Conv1d(8, D, kernel_size=3, padding=1)   # 不加入 optimizer

    def fit(self, mf_tr):
        sc = np.zeros(8, np.float64); ssq = sc.copy(); cnt = sc.copy()
        for _, row in mf_tr.iterrows():
            for col in ["neg_wav", "neu_wav", "pos_wav"]:
                A = ext_feat(*load_wav(row[col]))
                for c in range(8): sc[c] += A[c].sum(); ssq[c] += (A[c]**2).sum(); cnt[c] += A[c].size
        raw_cnt = cnt.copy()
        cnt = np.maximum(cnt, 1); m = sc / cnt; v = np.maximum(ssq / cnt - m**2, 1e-6)
        self.mean = torch.tensor(m, dtype=torch.float32)
        self.std  = torch.tensor(v**0.5, dtype=torch.float32)
        self._fit = True
        _names = ["mfcc_mean","mfcc_std","chroma_mean","chroma_std",
                  "mel_mean","mel_std","contrast_mean","contrast_std"]
        print("AudioNorm (per-channel)")
        for _c, _n in enumerate(_names):
            print(f"  channel {_c} ({_n}): mean={self.mean[_c]:.4f}, std={self.std[_c]:.4f}, count={int(raw_cnt[_c])}")

    def encode(self, row):
        AL = []
        for col in ["neg_wav", "neu_wav", "pos_wav"]:
            A = torch.tensor(ext_feat(*load_wav(row[col])), dtype=torch.float32)
            if self._fit: A = (A - self.mean.view(-1,1)) / (self.std.view(-1,1) + 1e-6)
            Ap = torch.nn.functional.adaptive_avg_pool1d(A.unsqueeze(0), L).squeeze(0)
            AL.append(self.conv(Ap.unsqueeze(0)).squeeze(0))
        return torch.stack(AL)

audio_encoder = AudioEncoder()
print("Fitting AudioEncoder normalizer on train set ...")
audio_encoder.fit(mf_tr)
print("over")
print()

In [ ]:
# Cell 6: 编码 samples -> Dataset -> DataLoader

def make_samples(mf_split, te, ae, split_name=""):
    samps = []; total = len(mf_split)
    print(f"Building samples for split='{split_name}', n={total}")
    for i, (_, row) in enumerate(mf_split.iterrows()):
        y = row["y_bin"]
        if pd.isna(y): continue
        with torch.no_grad():
            T, mask = te.encode(row); A = ae.encode(row)
        samps.append({"subject_id": row["subject_id"],
                      "T": T.float().cpu(), "A": A.float().cpu(),
                      "m": mask.bool().cpu(), "y": int(y)})
        if (i + 1) == total:
            print(f"  [{split_name}] encoded {i+1}/{total} subjects")
    return samps

class EATD(Dataset):
    def __init__(self, s): self.s = s
    def __len__(self): return len(self.s)
    def __getitem__(self, i): return self.s[i]

def collate(batch):
    T = torch.stack([b["T"] for b in batch])
    A = torch.stack([b["A"] for b in batch])
    M = torch.stack([b["m"] for b in batch])
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    return {"T": T.to(device), "A": A.to(device), "m": M.to(device), "y": y.to(device)}

def make_loader(samps, shuffle):
    return DataLoader(EATD(samps), BATCH_SIZE, shuffle=shuffle, num_workers=0, collate_fn=collate)

tr_s = make_samples(mf_tr, text_encoder, audio_encoder, "train")
va_s = make_samples(manifest[manifest["split"] == "val"].reset_index(drop=True),
                    text_encoder, audio_encoder, "val")
train_loader = make_loader(tr_s, shuffle=True)
val_loader   = make_loader(va_s, shuffle=False)

print()
print("samples built")
print(f"  len(train_samples) = {len(tr_s)}")
print(f"  len(val_samples)   = {len(va_s)}")
print()
print("Sample[0] shapes & types:")
_s0 = tr_s[0]
print(f"  subject_id : {_s0['subject_id']}")
print(f"  T.shape    : {_s0['T'].shape}")
print(f"  A.shape    : {_s0['A'].shape}")
print(f"  mask.shape : {_s0['m'].shape}")
print(f"  y          : {_s0['y']}  {type(_s0['y'])}")


In [ ]:
# Cell 7: SAG-Fusion 模型

class GFB(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.c = nn.Conv1d(2*d, d, 3, padding=1)
    def forward(self, T, A):
        B, S, D_, L_ = T.shape
        x = torch.cat([A, T], 2).view(B*S, 2*D_, L_)
        g = torch.sigmoid(self.c(x)).view(B, S, D_, L_)
        return g * A + (1 - g) * T

class CMA(nn.Module):
    def __init__(self, d, h=2, drop=0.1):
        super().__init__()
        self.mha  = nn.MultiheadAttention(d, h, dropout=drop, batch_first=True)
        self.ln   = nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)
    def forward(self, T, A, mask, F_):
        B, S, D_, L_ = T.shape
        Ts = T.permute(0,1,3,2).reshape(B*S, L_, D_)
        As = A.permute(0,1,3,2).reshape(B*S, L_, D_)
        Fs = F_.permute(0,1,3,2).reshape(B*S, L_, D_)
        kpm = (mask.view(B*S, L_) == 0)
        ao, _ = self.mha(Ts, As, As, key_padding_mask=kpm)
        out = self.ln(Fs + self.drop(ao))
        return out.reshape(B, S, L_, D_).permute(0,1,3,2).contiguous()

class Head(nn.Module):
    def __init__(self, d, nc=2, drop=0.1):
        super().__init__()
        self.c  = nn.Conv1d(d, d, 3, padding=1)
        self.af = nn.Linear(d, 1, bias=False)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(d, nc)
    def forward(self, Z, mask):
        B, S, D_, L_ = Z.shape
        z  = Z.view(B*S, D_, L_)
        z  = torch.relu(self.c(z)).view(B, S, D_, L_).mean(1)
        ms = mask.any(1).float()
        zt = z.permute(0, 2, 1)
        sc = self.af(zt).squeeze(-1)
        sc = sc.masked_fill(ms == 0, -1e9)
        at = torch.softmax(sc, 1).unsqueeze(-1)
        vec = (zt * at).sum(1)
        return self.fc(self.drop(vec)), at.squeeze(-1)

class SAGFusion(nn.Module):
    def __init__(self, d=D, drop=DROPOUT_P):
        super().__init__()
        self.gate  = GFB(d)
        self.cross = CMA(d, 2, drop)
        self.head  = Head(d, 2, drop)
    def forward(self, T, A, mask):
        F_ = self.gate(T, A)
        Z  = self.cross(T, A, mask, F_)
        return self.head(Z, mask)

set_seed(MODEL_SEED)
model = SAGFusion(D, DROPOUT_P).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params:,}")


In [ ]:
# Cell 8: 损失 & 优化器 & 训练/评估工具函数

import copy
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

class_weights = torch.tensor(CLASS_WEIGHT, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def get_probs_targets(m, loader):
    m.eval(); probs, ys = [], []
    with torch.no_grad():
        for b in loader:
            lg, _ = m(b["T"], b["A"], b["m"])
            probs.append(torch.softmax(lg, 1)[:,1].cpu())
            ys.append(b["y"].cpu())
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()

def prf_at_thr(probs, targets, thr):
    preds = (probs >= thr).astype(int)
    tp = ((preds==1)&(targets==1)).sum(); fp = ((preds==1)&(targets==0)).sum()
    fn = ((preds==0)&(targets==1)).sum(); tn = ((preds==0)&(targets==0)).sum()
    prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
    f1   = 2*prec*rec/max(prec+rec,1e-12)
    acc  = (tp+tn)/max(len(targets),1)
    return float(acc), float(prec), float(rec), float(f1)

def best_threshold(probs, targets):
    best = {"thr":0.5, "f1":-1, "acc":0, "prec":0, "rec":0}
    for thr in THRESHOLDS:
        acc, p, r, f1 = prf_at_thr(probs, targets, thr)
        if f1 > best["f1"]: best = {"thr":thr, "f1":f1, "acc":acc, "prec":p, "rec":r}
    return best

def train_one(m, loader, opt):
    m.train(); ls = 0; n = 0
    for b in loader:
        opt.zero_grad()
        lg, _ = m(b["T"], b["A"], b["m"])
        loss = criterion(lg, b["y"]); loss.backward(); opt.step()
        ls += loss.item() * b["y"].size(0); n += b["y"].size(0)
    return ls / max(n, 1)

def plot_confusion_matrix(m, loader, thr, title="Confusion Matrix"):
    m.eval()
    probs, tgts = get_probs_targets(m, loader)
    preds = (probs >= thr).astype(int)
    cm = confusion_matrix(tgts, preds)
    acc, prec, rec, f1 = prf_at_thr(probs, tgts, thr)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Dep", "Depressed"],
                yticklabels=["Non-Dep", "Depressed"])
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(f"{title}\nAcc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    plt.tight_layout(); plt.show()
    print(f"Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}")
    return cm

print("cell 8 over")


In [ ]:
# Cell 9: 训练主循环
# 注：深度学习模型在小样本数据上受随机初始化影响，不同运行环境结果可能有轻微偏差

best_val_f1 = -1.0
best_epoch  = 0
best_state  = copy.deepcopy(model.state_dict())
no_improve  = 0

print("Start Training")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one(model, train_loader, optimizer)

    va_probs, va_tgts = get_probs_targets(model, val_loader)
    res = best_threshold(va_probs, va_tgts)
    vf1 = res["f1"]

    print(f"[Epoch {epoch:>3}/{NUM_EPOCHS}]  loss={train_loss:.4f}  "
          f"val_f1={vf1:.4f}  thr={res['thr']}  "
          f"acc={res['acc']:.4f}  p={res['prec']:.4f}  r={res['rec']:.4f}")

    if vf1 > best_val_f1 + 1e-4:
        best_val_f1 = vf1
        best_epoch  = epoch
        best_state  = copy.deepcopy(model.state_dict())
        no_improve  = 0
        torch.save({
            "epoch":       best_epoch,
            "val_f1":      best_val_f1,
            "model_state": best_state,
        }, CHECKPOINT_PATH)
        print(f"  --> best val_f1={best_val_f1:.4f} @ epoch {best_epoch}  [saved]")
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"early stopping at epoch {epoch}, best={best_val_f1:.4f} @ {best_epoch}")
        break

model.load_state_dict(best_state)
print("=" * 60)
print(f"done.  best val_f1={best_val_f1:.4f}  best_epoch={best_epoch}")


In [ ]:
# Cell 10: Val 集评估 & 混淆矩阵

print(f"Val evaluation  thr={FIXED_THR}")
cm = plot_confusion_matrix(model, val_loader, FIXED_THR,
                           title="SAG-Fusion Val")
